## 1. Setup & Imports
All third-party imports, device selection, and global random seed.

In [ ]:
import os, random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 2. Configuration
All hyperparameters and paths in one place — change here only.

In [ ]:
cfg = SimpleNamespace(
    # --- Paths ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small",
    output_dir="/kaggle/working",
    # --- Model ---
    input_dim=384,
    latent_dim=64,
    output_dim=5,
    dropout=0.1,
    # --- Augmentation ---
    # Subset of ["identity", "hflip", "vflip", "hflip_vflip"]; must include "identity"
    augmentations=["identity", "hflip", "vflip", "hflip_vflip"],
    # --- Training ---
    epochs=80,
    lr=3e-4,
    weight_decay=1e-3,
    batch_size=64,
    seed=42,
    # --- Split ---
    n_splits=5,
    val_fold=0,
)

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV     = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV      = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("train.csv:",  os.path.exists(TRAIN_CSV))
print("test.csv:",   os.path.exists(TEST_CSV))
print("train dir:",  os.path.exists(TRAIN_IMG_DIR))
print("test dir:",   os.path.exists(TEST_IMG_DIR))
print("dino dir:",   os.path.exists(cfg.dino_weights_dir))

## 3. Load DINOv2
Load ViT-S/14 from uploaded HuggingFace weights; smoke-test at 504×252 resolution.

In [ ]:
print(f"Loading DINOv2 ViT-S/14 from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

_dummy = torch.zeros(1, 3, 252, 504, device=device)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed — CLS shape: {_cls.shape}")
del _dummy, _out, _cls

## 4. Load Training CSV
Pivot long-format CSV to wide format (one row per image) and extract Y matrix.

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all = df_wide[TARGETS].values.astype(np.float32)  # (N, 5)
train_image_ids_all = df_wide["image_id"].values

print(f"Training images: {len(df_wide)}")
print(f"Y_all shape:     {Y_all.shape}")
print(df_wide.head(3))

## 5. Extract DINOv2 Features — Training Images
Preprocess at 504×252 and extract CLS tokens with 4-orientation augmentation (identity, hflip, vflip, hflip+vflip).

In [ ]:
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_features(image_paths, model, transform):
    """Return (N, 384) float32 array of CLS-token features."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(pixel_values=x, interpolate_pos_encoding=True)
            feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
        feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


_AUGMENTATION_FNS = {
    "identity":    lambda img: img,
    "hflip":       lambda img: TF.hflip(img),
    "vflip":       lambda img: TF.vflip(img),
    "hflip_vflip": lambda img: TF.hflip(TF.vflip(img)),
}
_ORIENTATIONS = [_AUGMENTATION_FNS[a] for a in cfg.augmentations]
n_aug = len(_ORIENTATIONS)


def extract_features_augmented(image_paths, model, transform):
    """Return (N*n_aug, 384) array — each image appears n_aug times."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        for flip_fn in _ORIENTATIONS:
            x = transform(flip_fn(img)).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(pixel_values=x, interpolate_pos_encoding=True)
                feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
            feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


train_image_paths = [
    os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg")
    for iid in train_image_ids_all
]
N_orig = len(train_image_paths)
print(f"Extracting features for {N_orig} training images ({n_aug} orientations each)...")
X_all = extract_features_augmented(train_image_paths, dino, _dino_transform)

Y_all               = np.repeat(Y_all,               n_aug, axis=0)
train_image_ids_all = np.repeat(train_image_ids_all, n_aug)
image_group_ids     = np.repeat(np.arange(N_orig),   n_aug)

print(f"X_all shape:         {X_all.shape}")
print(f"Y_all shape:         {Y_all.shape}")
print(f"Unique groups:       {len(np.unique(image_group_ids))}")

assert X_all.shape == (N_orig * n_aug, 384)
assert Y_all.shape == (N_orig * n_aug, 5)
assert len(image_group_ids) == N_orig * n_aug
assert len(np.unique(image_group_ids)) == N_orig
assert np.all(np.bincount(image_group_ids) == n_aug)
print("Sanity checks passed.")

## 6. Extract DINOv2 Features — Test Images
Same preprocessing pipeline applied to the held-out test set.

In [ ]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids = df_test_unique["image_id"].values
test_image_paths = [
    os.path.join(TEST_IMG_DIR, f"{iid}.jpg")
    for iid in test_image_ids
]

print(f"Extracting features for {len(test_image_paths)} test images...")
X_test = extract_features(test_image_paths, dino, _dino_transform)
print(f"X_test shape: {X_test.shape}")

## 7. GroupKFold Train/Val Split
Fold 0 of a 5-fold GroupKFold; groups on original image index so all 4 orientations of an image land in the same fold.

In [ ]:
gkf = GroupKFold(n_splits=cfg.n_splits)
splits = list(gkf.split(X_all, groups=image_group_ids))
train_idx, val_idx = splits[cfg.val_fold]

X_train     = X_all[train_idx]
Y_train_raw = Y_all[train_idx]
X_val       = X_all[val_idx]
Y_val_raw   = Y_all[val_idx]

train_groups = set(image_group_ids[train_idx].tolist())
val_groups   = set(image_group_ids[val_idx].tolist())
assert train_groups.isdisjoint(val_groups), "Group leakage!"
print("Group overlap check: PASSED (zero overlap)")

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(f"Y_train range: [{Y_train_raw.min():.1f}, {Y_train_raw.max():.1f}]")

## 8. Model Architecture — Encoder + MLP Head
Encoder 384→256→64, Head 64→32→5.

In [ ]:
class Encoder(nn.Module):
    """384 → 256 → 64 (GELU, Dropout)."""

    def __init__(self, input_dim=384, latent_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class Head(nn.Module):
    """64 → 32 → 5 (GELU, Dropout)."""

    def __init__(self, latent_dim=64, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class BiomassModel(nn.Module):

    def __init__(self, input_dim=384, latent_dim=64, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head = Head(latent_dim, output_dim, dropout)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.head(self.encode(x))


print("BiomassModel defined.")
_tmp = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f"Parameters: {n_params:,}")
del _tmp

## 9. Loss, Metrics & Dataset
Weighted SmoothL1, weighted global R², per-target RMSE, TwoNN ID estimate, and a minimal PyTorch Dataset.

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)


def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()


def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


def twonn_inhouse(X):
    """In-house TwoNN (Facco et al. 2017). Returns float ID estimate."""
    nn_model = NearestNeighbors(n_neighbors=3, algorithm="auto", metric="euclidean")
    nn_model.fit(X)
    dists, _ = nn_model.kneighbors(X)
    r1, r2   = dists[:, 1], dists[:, 2]
    mask     = r1 > 0
    r1, r2   = r1[mask], r2[mask]
    mu       = r2 / r1
    mu_s     = np.sort(mu)
    n        = len(mu_s)
    ecdf     = np.arange(1, n + 1) / n
    log_mu   = np.log(mu_s)
    log_surv = -np.log1p(-ecdf + 1e-10)
    return float(np.dot(log_mu, log_surv) / np.dot(log_mu, log_mu))


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    r2   = weighted_global_r2(y_true, y_pred)
    rmse = rmse_per_target(y_true, y_pred)
    return total_loss / len(loader.dataset), r2, rmse, y_pred, y_true


print("Loss, metrics, dataset, eval_epoch defined.")

## 10. Model Training — Baseline
Standard minibatch training loop with AdamW + cosine LR schedule; best checkpoint tracked by val R².

In [ ]:
def train_baseline_loop(model, train_ds, val_ds, epochs, lr, weight_decay, batch_size,
                        seed, train_device, verbose=True):
    """Standard minibatch training loop. Returns (history, best_state_dict)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_ldr = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_ldr   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr / 100)

    history = {"train_loss": [], "val_loss": [], "val_r2": [], "val_rmse": []}
    best_val_r2 = -float("inf")
    best_state  = None

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss, n_steps = 0.0, 0
        for X_b, y_b in train_ldr:
            X_b, y_b = X_b.to(train_device), y_b.to(train_device)
            optimizer.zero_grad()
            pred = model(X_b)
            loss = _weighted_smooth_l1(pred, y_b, _LOSS_WEIGHTS)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * len(X_b)
            n_steps += len(X_b)

        scheduler.step()

        tr = ep_loss / n_steps
        val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_ldr, train_device)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(tr)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)
        history["val_rmse"].append(val_rmse)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            rmse_str = "  ".join(
                f"{t.split('_')[1]}:{v:.2f}" for t, v in val_rmse.items()
            )
            print(
                f"  ep {epoch:3d}  tr={tr:.4f}  "
                f"val_loss={val_loss:.4f}  val_R²={val_r2:.4f}  [{rmse_str}]"
            )

    return history, best_state


train_ds = BiomassDataset(X_train, Y_train_raw)
val_ds   = BiomassDataset(X_val,   Y_val_raw)

torch.manual_seed(cfg.seed)
model = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout).to(device)

history, best_state = train_baseline_loop(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=cfg.epochs,
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    batch_size=cfg.batch_size,
    seed=cfg.seed,
    train_device=device,
    verbose=True,
)

best_epoch = int(np.argmax(history["val_r2"])) + 1
print(f"\nBest val R² = {max(history['val_r2']):.4f} at epoch {best_epoch}")

## 11. Latent ID Estimation
Estimate intrinsic dimension on the trained 32-d latents via TwoNN.

In [ ]:
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()

with torch.no_grad():
    latents_train = model.encode(
        torch.tensor(X_train, dtype=torch.float32, device=device)
    ).cpu().numpy()

d_latent = twonn_inhouse(latents_train)
print(f"Latent ID (TwoNN, 64-d space): {d_latent:.2f}")
if d_latent < 3 or d_latent > 10:
    print("  WARNING: ID outside expected 3-10 range.")
else:
    print("  ID looks reasonable (expected ~4-5).")

## 12. Final Validation Metrics
Evaluate best checkpoint; flag if weighted R² < 0.75.

In [ ]:
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
_, val_r2_best, val_rmse_best, val_preds, val_true = eval_epoch(model, val_loader, device)

print("=" * 55)
print(f"Final val weighted R²: {val_r2_best:.4f}")
print("=" * 55)
print("RMSE per target:")
for t, v in val_rmse_best.items():
    print(f"  {t:<18}: {v:.4f}")
print("=" * 55)

if val_r2_best < 0.75:
    print(
        f"\n  *** WARNING: val R² = {val_r2_best:.4f} is below the expected baseline."
        "\n      Check DINOv2 weights / resize / normalisation."
    )
else:
    print(f"\n  R² = {val_r2_best:.4f} — looks reasonable.")

## 13. Test Inference
Encode test features through encoder+head; clip negative predictions.

In [ ]:
model.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    test_preds = model(X_test_t).cpu().numpy()

test_preds = np.clip(test_preds, 0.0, None)

print(f"test_preds shape: {test_preds.shape}")
print(f"test_preds range: [{test_preds.min():.2f}, {test_preds.max():.2f}]")
print("Sample predictions (first 3 test images):")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds[i].round(2)))}")

## 14. Format Predictions as Submission CSV
Map (image_id, 5 targets) → long-format rows matching test.csv sample_id keys.

In [ ]:
def prepare_submission(test_csv_path, predictions, image_ids):
    df_t = pd.read_csv(test_csv_path)

    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }

    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        return max(0.0, pred_dict.get(img_id, {}).get(target_name, 0.0))

    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]


submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)

out_path = os.path.join(cfg.output_dir, "submission.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))